In [ ]:
import sys
sys.path.append('/home/jovyan/work/Similarity-Aware-Label-Smoothing')

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNet18, CifarDenseNet121, TinyResNet34, TinyDenseNet121
from metrics import calibration_errors, nll_loss
import random
import os
from scipy.stats import spearmanr, wilcoxon



## Hyperparams

In [5]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [8]:
dataset = "cifar100"
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 200
num_classes = 100
epsilon = 0.05
K = 5

## Training Utils

In [9]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [10]:
path = f"Similarity-Aware-Label-Smoothing/scores/4_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=K, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [17]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [18]:
classwise_target = uniform_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target[0])
print(classwise_target.shape)

tensor([9.5000e-01, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04, 5.0505e-04,
        5.0505e-04, 5.0505e-04, 5.0505e-

In [19]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [20]:
model = CifarDenseNet121(num_classes=num_classes).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, total_iters=5
        ),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=195
        )
    ],
    milestones=[5]
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

  0%|          | 0/391 [00:00<?, ?it/s]

Epoch 1/200 | Loss: 3.9549 | Test Acc: 0.1700 | Top-5 Test Acc: 0.4294


Epoch 2/200 | Loss: 3.5154 | Test Acc: 0.2484 | Top-5 Test Acc: 0.5403


Epoch 3/200 | Loss: 3.1937 | Test Acc: 0.2863 | Top-5 Test Acc: 0.5990


Epoch 4/200 | Loss: 2.9255 | Test Acc: 0.3109 | Top-5 Test Acc: 0.6197


...

Epoch 196/200 | Loss: 0.4517 | Test Acc: 0.6818 | Top-5 Test Acc: 0.8726


Epoch 197/200 | Loss: 0.4516 | Test Acc: 0.6821 | Top-5 Test Acc: 0.8736


Epoch 198/200 | Loss: 0.4514 | Test Acc: 0.6817 | Top-5 Test Acc: 0.8727


Epoch 199/200 | Loss: 0.4517 | Test Acc: 0.6811 | Top-5 Test Acc: 0.8741


Epoch 200/200 | Loss: 0.4516 | Test Acc: 0.6805 | Top-5 Test Acc: 0.8731


In [21]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)
print()
print("ECE:", calibration_errors(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))
test_acc = accuracy(model, test_loader)
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")
print()



ECE: (0.07243743538856506, 0.21079373359680176)
NLL: 1.3904963731765747
Top-1 Test Acc: 0.6805 | Top-5 Test Acc: 0.8731



In [22]:
PATH = f"vae8_{'0'+str(epsilon).removeprefix("0.")}_{K}_{seed}.pth"
torch.save(model.state_dict(), PATH)